# All-year `model_df` merge (2020–2023)

Builds a **person–year** modeling table for each MEPS year (same construction as `2023_clean.ipynb` `model_df`), writes each year out, then stacks them into one panel.

**Grain after merge:** one row per `(DUPERSID, YEAR)` — not one row per person across the whole panel.
#the model is going to duplicate user id - everything is duplicated

## Design decisions & assumptions

### 1. Years are built independently, then stacked

Each year runs the same adherence pipeline (`clean_meps.build`) with **that year’s** Rx file, CLNK, conditions, and person file.

| Piece | Scope |
|---|---|
| Numerator (`total_valid_days`) | Sum of `RXDAYSUP` for fills **in that calendar year**, capped at `min(365, total_days_supply)` |
| Denominator (`total_days_supply`) | Person’s PSTATS-eligible days **in that year**, further clamped by drug-start days **in that year** |
| `meps_adherence_ratio` | `100 * numerator / denominator` for that year only |

We do **not** invent a multi-year denominator (e.g. 1460 days) or average ratios across years before stacking.

### 2. Same patient in more than one year

**Assumption:** MEPS `DUPERSID` is stable for a person across years they remain in the survey. A person present in 2021 and 2022 contributes **two rows** in the merged panel:

- `(DUPERSID=X, YEAR=2021)` with 2021 fills / 2021 PSTATS window / 2021 demographics
- `(DUPERSID=X, YEAR=2022)` with 2022 fills / 2022 PSTATS window / 2022 demographics

Demographics (`AGE`, insurance, poverty, delay flags) are **year-specific** (taken from that year’s person file). They are allowed to change across rows for the same `DUPERSID`.

### 3. Denominator when a drug starts mid-year (or between rounds)

Within a year, denominator is:

```text
total_days_supply = min(person_PSTATS_days_in_year, drug_start_days)
```

`drug_start_days` (from earliest fill’s `RXBEGYRX` / `RXBEGMM`):

| Drug start | `drug_start_days` |
|---|---|
| Before this calendar year | **365** (drug considered active all year; PSTATS may still shrink the window) |
| This year, month 1–12 known | Days from **1st of that month → Dec 31** |
| This year, month missing / sentinel | **365** (conservative — do not invent a start month) |
| Start year missing or after this year | **365** (conservative) |

**Between years:** if a patient starts a drug in Nov 2021, then:

- **2021 row:** short `drug_start_days` (~61 days from Nov 1), so adherence is not punished for “missing” Jan–Oct.
- **2022 row:** `RXBEGYRX < 2022` → `drug_start_days = 365`; denominator is the **2022** PSTATS window only. 2021 fills never enter the 2022 numerator.

**Between rounds inside a year:** PSTATS / BEGRF / ENDRF already define the person’s eligible window for that year (joiners, leavers, R5/3 nonresponse). Drug-start clamp is applied **on top of** that window. We do not re-open eligibility across year boundaries.

### 4. What we intentionally do *not* do

- No carry-forward of unused days-supply from Dec Year N into Year N+1.
- No pooled 2020–2023 MPR with a single long denominator.
- No collapsing multi-year patients into one row (that would mix incompatible denominators).
- RX / ICD one-hots are built **per year**, then column-aligned on merge (`fillna(0)` for drugs/conditions unseen in a year).

### 5. Outputs

| Path | Contents |
|---|---|
| `Notebooks/MEPS/output/{year}/tables/model_df_pair.parquet` | Drug–condition pair rows + demos (pre RX/ICD one-hot) |
| `Notebooks/MEPS/output/{year}/tables/model_df.parquet` | Patient–year rows (post one-hot) |
| `Notebooks/MEPS/output/all_years/tables/model_df_all_years.parquet` | Stacked patient–year panel |

In [10]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

# Repo layout: Notebooks/allYearMergeClean.ipynb → repo root has app/ and data/
REPO = Path("..").resolve()
if not (REPO / "app" / "clean_meps.py").exists():
    REPO = Path.cwd().resolve()
    if not (REPO / "app" / "clean_meps.py").exists():
        REPO = Path.cwd().resolve().parent

sys.path.insert(0, str(REPO / "app"))

from clean_meps import (  # noqa: E402
    YEAR_FILES,
    all_years_output_dirs,
    build,
    output_dirs,
    resolve_meps_dir,
)

MEPS_DIR = resolve_meps_dir()
YEARS = [2020, 2021, 2022, 2023]
pd.set_option("display.max_columns", 40)
print("MEPS_DIR:", MEPS_DIR)
print("REPO:", REPO)

MEPS_DIR: /Users/friana/Medical Adherence/data/MEPS
REPO: /Users/friana/Medical Adherence


## Step A — Worked examples (denominator logic, no MEPS I/O)

These synthetic cases mirror `_drug_start_days_in_year` + `min(PSTATS, drug_start)` from `clean_meps`.

In [11]:
from datetime import date


def drug_start_days(year: int, rxbegyrx, rxbegmm) -> int:
    """Same rules as clean_meps._drug_start_days_in_year (scalar version)."""
    fy = pd.to_numeric(rxbegyrx, errors="coerce")
    fm = pd.to_numeric(rxbegmm, errors="coerce")
    if pd.isna(fy) or fy != year:
        return 365
    if pd.isna(fm) or not (1 <= fm <= 12):
        return 365
    year_end = date(year, 12, 31)
    return (year_end - date(year, int(fm), 1)).days + 1


def year_denom(pstats_days: int, year: int, rxbegyrx, rxbegmm) -> int:
    return int(min(pstats_days, drug_start_days(year, rxbegyrx, rxbegmm)))


examples = pd.DataFrame([
    # Multi-year patient, drug started before panel
    {"case": "same patient, continuing drug",
     "year": 2021, "pstats": 365, "RXBEGYRX": 2019, "RXBEGMM": 3,
     "note": "Prior-year start → drug_start=365; denom=PSTATS"},
    {"case": "same patient, continuing drug",
     "year": 2022, "pstats": 365, "RXBEGYRX": 2019, "RXBEGMM": 3,
     "note": "New year: still drug_start=365; 2021 fills excluded"},
    # Started mid-year, then present next year
    {"case": "start Nov Y1, continue Y2",
     "year": 2021, "pstats": 365, "RXBEGYRX": 2021, "RXBEGMM": 11,
     "note": "Mid-year start shortens 2021 denom only"},
    {"case": "start Nov Y1, continue Y2",
     "year": 2022, "pstats": 365, "RXBEGYRX": 2021, "RXBEGMM": 11,
     "note": "2022 treats drug as full-year eligible (start < 2022)"},
    # Joiner mid-year (PSTATS short) + drug start same month
    {"case": "joined July, drug starts July",
     "year": 2022, "pstats": 184, "RXBEGYRX": 2022, "RXBEGMM": 7,
     "note": "min(184, days from Jul 1)=184 — PSTATS binds"},
    # Drug starts after person became eligible
    {"case": "full-year person, drug starts Sept",
     "year": 2022, "pstats": 365, "RXBEGYRX": 2022, "RXBEGMM": 9,
     "note": "min(365, days from Sep 1) — drug-start binds"},
    # Missing month sentinel
    {"case": "start year known, month -1",
     "year": 2023, "pstats": 365, "RXBEGYRX": 2023, "RXBEGMM": -1,
     "note": "Conservative: drug_start=365 when month unknown"},
])

examples["drug_start_days"] = [
    drug_start_days(r.year, r.RXBEGYRX, r.RXBEGMM) for r in examples.itertuples()
]
examples["total_days_supply"] = [
    year_denom(r.pstats, r.year, r.RXBEGYRX, r.RXBEGMM) for r in examples.itertuples()
]
examples

,case,year,pstats,RXBEGYRX,RXBEGMM,note,drug_start_days,total_days_supply
0,"same patient, continuing drug",2021,365,2019,3,Prior-year start → drug_start=365; denom=PSTATS,365,365
1,"same patient, continuing drug",2022,365,2019,3,New year: still drug_start=365; 2021 fills exc...,365,365
2,"start Nov Y1, continue Y2",2021,365,2021,11,Mid-year start shortens 2021 denom only,61,61
3,"start Nov Y1, continue Y2",2022,365,2021,11,2022 treats drug as full-year eligible (start ...,365,365
4,"joined July, drug starts July",2022,184,2022,7,"min(184, days from Jul 1)=184 — PSTATS binds",184,184
5,"full-year person, drug starts Sept",2022,365,2022,9,"min(365, days from Sep 1) — drug-start binds",122,122
6,"start year known, month -1",2023,365,2023,-1,Conservative: drug_start=365 when month unknown,365,365


## Step B — Helpers: build each year’s `model_df` (same recipe as 2023 notebook)

In [12]:
def load_delay(year: int) -> pd.DataFrame:
    """DLAYPM42 / DLAYCA42 live on the person file; build() does not attach them."""
    person = pd.read_excel(
        MEPS_DIR / YEAR_FILES[year]["person"],
        engine="calamine",
        usecols=["DUPERSID", "DLAYPM42", "DLAYCA42"],
    )
    return person.drop_duplicates("DUPERSID")


def build_pair_model_df(gm: pd.DataFrame, year: int, delay: pd.DataFrame) -> pd.DataFrame:
    """Pair-level model_df ≈ 2023_clean cells 67–86 (before RX/ICD one-hot)."""
    yy = f"{year % 100:02d}"
    df = gm.drop(columns=["DLAYPM42", "DLAYCA42"], errors="ignore").merge(
        delay, on="DUPERSID", how="left"
    )
    # Prefer primary_* from clean_meps; avoid duplicate ICD10CDX column names
    if "primary_ICD10CDX" in df.columns:
        if "ICD10CDX" in df.columns:
            df = df.drop(columns=["ICD10CDX"])
        df = df.rename(columns={"primary_ICD10CDX": "ICD10CDX"})
    if "primary_ICD10CDX_LABEL" in df.columns:
        if "ICD10CDX_LABEL" in df.columns:
            df = df.drop(columns=["ICD10CDX_LABEL"])
        df = df.rename(columns={"primary_ICD10CDX_LABEL": "ICD10CDX_LABEL"})

    renames = {
        f"AGE{yy}X": "AGE",
        f"INSCOV{yy}": "INSCOV",
        f"POVCAT{yy}": "POVCAT",
        f"FAMINC{yy}": "FAMINC",
        f"RXSF{yy}X": "RXSF_PATIENT",
        f"RXXP{yy}X": "RXXP_TOTAL",
    }
    df = df.rename(columns={k: v for k, v in renames.items() if k in df.columns})

    keep = [
        "DUPERSID", "DRUGIDX", "RXNAME", "ICD10CDX", "ICD10CDX_LABEL",
        "AGE", "SEX", "INSCOV", "POVCAT", "FAMINC", "RACEV2X",
        "RXSF_PATIENT", "RXXP_TOTAL", "DLAYPM42", "DLAYCA42",
        "meps_adherence_ratio", "total_days_supply", "drug_start_days",
        "RXBEGYRX", "first_month", "chronic_conditions", "n_chronic_conditions",
        "TC1", "TC1S1",
    ]
    model = df[[c for c in keep if c in df.columns]].copy()
    model = model.loc[:, ~model.columns.duplicated()].copy()
    model["YEAR"] = year

    for c in ("RXSF_PATIENT", "RXXP_TOTAL"):
        if c in model.columns:
            model[c] = pd.to_numeric(model[c], errors="coerce").fillna(0).clip(lower=0)

    model["is_adherent"] = np.where(model["meps_adherence_ratio"] >= 60, 1, 0)

    skip = {
        "DUPERSID", "DRUGIDX", "RXNAME", "ICD10CDX", "ICD10CDX_LABEL",
        "chronic_conditions", "YEAR",
    }
    feature_cols = [c for c in model.columns if c not in skip]
    for c in feature_cols:
        model[c] = pd.to_numeric(model[c], errors="coerce")
    model = model.loc[~model[feature_cols].lt(0).any(axis=1)].copy()

    model["INSCOV_PRIVATE"] = (model["INSCOV"] == 1).astype(int)
    model["INSCOV_PUBLIC"] = (model["INSCOV"] == 2).astype(int)
    model["INSCOV_UNINSURED"] = (model["INSCOV"] == 3).astype(int)
    model = model.drop(columns=["INSCOV"])

    model["MALE"] = np.where(model["SEX"] == 1, 1, 0)
    model["FEMALE"] = np.where(model["SEX"] == 2, 1, 0)
    model = model.drop(columns=["SEX"])

    model["WHITE"] = np.where(model["RACEV2X"] == 1, 1, 0)
    model["BLACK"] = np.where(model["RACEV2X"] == 2, 1, 0)
    model["AMER_INDIAN"] = np.where(model["RACEV2X"] == 3, 1, 0)
    model["ASIAN_INDIAN"] = np.where(model["RACEV2X"] == 4, 1, 0)
    model["CHINESE"] = np.where(model["RACEV2X"] == 5, 1, 0)
    model["FILIPINO"] = np.where(model["RACEV2X"] == 6, 1, 0)
    model = model.drop(columns=["RACEV2X"])

    model["PMED_DELAY_COST"] = np.where(model["DLAYPM42"] == 1, 1, 0)
    model["NO_PMED_DELAY_COST"] = np.where(model["DLAYPM42"] == 2, 1, 0)
    model["CARE_DELAY_COST"] = np.where(model["DLAYCA42"] == 1, 1, 0)
    model["NO_CARE_DELAY_COST"] = np.where(model["DLAYCA42"] == 2, 1, 0)
    model = model.drop(columns=["DLAYPM42", "DLAYCA42"])

    model["POV_POOR"] = np.where(model["POVCAT"] == 1, 1, 0)
    model["POV_NEAR_POOR"] = np.where(model["POVCAT"] == 2, 1, 0)
    model["POV_LOW"] = np.where(model["POVCAT"] == 3, 1, 0)
    model["POV_MIDDLE"] = np.where(model["POVCAT"] == 4, 1, 0)
    model["POV_HIGH"] = np.where(model["POVCAT"] == 5, 1, 0)
    model = model.drop(columns=["POVCAT", "ICD10CDX_LABEL"], errors="ignore")
    return model


def to_patient_year(pair: pd.DataFrame) -> pd.DataFrame:
    """Patient–YEAR grain with RX/ICD one-hots (≈ 2023 cells 87–88)."""
    rx_dummies = pd.get_dummies(pair["RXNAME"], prefix="RX").astype(int)
    if "chronic_conditions" in pair.columns:
        icd_dummies = (
            pair["chronic_conditions"].fillna("").str.get_dummies(sep=",")
            .add_prefix("ICD_").astype(int)
        )
        icd_dummies = icd_dummies.drop(
            columns=[c for c in ["ICD_"] if c in icd_dummies.columns]
        )
        drop_extra = ["RXNAME", "ICD10CDX", "chronic_conditions"]
    else:
        icd_dummies = pd.get_dummies(pair["ICD10CDX"], prefix="ICD").astype(int)
        drop_extra = ["RXNAME", "ICD10CDX"]

    drop_also = [
        "DRUGIDX", "TC1", "TC1S1", "first_month", "RXBEGYRX",
        "drug_start_days", "total_days_supply", "n_chronic_conditions",
    ]
    base = pair.drop(
        columns=[c for c in drop_extra + drop_also if c in pair.columns]
    )
    pair_oh = pd.concat([base, rx_dummies, icd_dummies], axis=1)

    rx_cols = [c for c in pair_oh.columns if c.startswith("RX_")]
    icd_cols = [c for c in pair_oh.columns if c.startswith("ICD_")]
    cost = {"RXSF_PATIENT", "RXXP_TOTAL"}
    keys = {"DUPERSID", "YEAR"}
    person_cols = [
        c for c in pair_oh.columns
        if c not in keys | {"meps_adherence_ratio"} | cost
        and c not in rx_cols and c not in icd_cols
    ]
    agg = {c: "max" for c in person_cols + rx_cols + icd_cols}
    agg["meps_adherence_ratio"] = "mean"
    agg["RXSF_PATIENT"] = "sum"
    agg["RXXP_TOTAL"] = "sum"

    out = pair_oh.groupby(["DUPERSID", "YEAR"], as_index=False).agg(agg)
    out["PATIENT_COST_SHARE"] = np.where(
        out["RXXP_TOTAL"] > 0,
        out["RXSF_PATIENT"] / out["RXXP_TOTAL"],
        0.0,
    )
    out = out.drop(columns=["RXSF_PATIENT", "RXXP_TOTAL"])
    out["is_adherent"] = np.where(out["meps_adherence_ratio"] >= 60, 1, 0)
    out["n_drugs"] = out[rx_cols].sum(axis=1)
    out["n_conditions"] = out[icd_cols].sum(axis=1)
    return out


def ensure_year_model_dfs(year: int, force: bool = False) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Build (or load cached) pair + patient–year model_dfs for one year."""
    tables, _ = output_dirs(year)
    tables.mkdir(parents=True, exist_ok=True)
    pair_path = tables / "model_df_pair.parquet"
    patient_path = tables / "model_df.parquet"
    raw_path = tables / "new_grouped_merge_df_chronic_drugs.parquet"

    if not force and pair_path.exists() and patient_path.exists():
        print(f"[{year}] cache hit")
        return pd.read_parquet(pair_path), pd.read_parquet(patient_path)

    t0 = time.time()
    print(f"[{year}] building via clean_meps.build …")
    gm, bridge, _log = build(year)
    age_col = f"AGE{year % 100:02d}X"
    if age_col in gm.columns:
        gm[age_col] = gm[age_col].astype("string")
    gm.to_parquet(raw_path, index=False)
    bridge.to_parquet(tables / "patient_drug_condition_bridge.parquet", index=False)

    delay = load_delay(year)
    pair = build_pair_model_df(gm, year, delay)
    if "AGE" in pair.columns:
        pair["AGE"] = pair["AGE"].astype("string")
    pair.to_parquet(pair_path, index=False)

    patient = to_patient_year(pair)
    if "AGE" in patient.columns:
        patient["AGE"] = patient["AGE"].astype("string")
    patient.to_parquet(patient_path, index=False)

    print(
        f"[{year}] wrote pair={len(pair):,} patient-year={len(patient):,} "
        f"in {time.time() - t0:.1f}s → {tables}"
    )
    return pair, patient


print("helpers ready")

helpers ready


## Step C — Build / load each year and write tables

First run is slow (~4–5 min/year reading Excel). Re-runs use parquet caches under `Notebooks/MEPS/output/{year}/tables/`.

In [13]:
pair_by_year: dict[int, pd.DataFrame] = {}
model_by_year: dict[int, pd.DataFrame] = {}

for y in YEARS:
    pair_by_year[y], model_by_year[y] = ensure_year_model_dfs(y)
    m = model_by_year[y]
    print(
        f"  YEAR={y}: patient-year rows={len(m):,} | "
        f"unique DUPERSID={m['DUPERSID'].nunique():,} | "
        f"adherent rate={m['is_adherent'].mean():.3f}"
    )

model_by_year[2023].head()

[2020] cache hit
  YEAR=2020: patient-year rows=7,718 | unique DUPERSID=7,718 | adherent rate=0.536
[2021] cache hit
  YEAR=2021: patient-year rows=8,377 | unique DUPERSID=8,377 | adherent rate=0.514
[2022] cache hit
  YEAR=2022: patient-year rows=6,770 | unique DUPERSID=6,770 | adherent rate=0.505
[2023] cache hit
  YEAR=2023: patient-year rows=5,826 | unique DUPERSID=5,826 | adherent rate=0.495


,DUPERSID,YEAR,AGE,FAMINC,is_adherent,INSCOV_PRIVATE,INSCOV_PUBLIC,INSCOV_UNINSURED,MALE,FEMALE,WHITE,BLACK,AMER_INDIAN,ASIAN_INDIAN,CHINESE,FILIPINO,PMED_DELAY_COST,NO_PMED_DELAY_COST,CARE_DELAY_COST,NO_CARE_DELAY_COST,...,ICD_M19,ICD_M35,ICD_M48,ICD_M50,ICD_M51,ICD_M53,ICD_M54,ICD_M81,ICD_M85,ICD_N18,ICD_N19,ICD_N40,ICD_R26,ICD_R32,ICD_U09,ICD_Z79,PATIENT_COST_SHARE,n_drugs,n_conditions,meps_adherence_ratio
0,2790002101,2023,58,130700,0,1,0,0,0,1,0,1,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.672668,1,1,41.095890
1,2790011101,2023,71,79240,1,1,0,0,1,0,1,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.739957,2,2,61.643836
2,2790011102,2023,71,79240,0,1,0,0,0,1,1,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.236679,2,2,32.809708
3,2790012101,2023,55,192950,1,1,0,0,0,1,0,1,0,0,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.033005,2,1,60.273973
4,2790012102,2023,54,192950,1,1,0,0,0,1,0,1,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000,1,1,65.753425


## Step D — Prove multi-year + mid-year-start behavior on real data

1. Find patients in ≥2 years.
2. Find a drug that **starts in year Y** (short `drug_start_days`) and continues in **Y+1** (`drug_start_days == 365`).

In [14]:
# Stack pair frames (keep denom diagnostics that patient-year drops)
pair_all = pd.concat([pair_by_year[y] for y in YEARS], ignore_index=True)

years_per_person = (
    pair_all.groupby("DUPERSID")["YEAR"].nunique().rename("n_years").reset_index()
)
multi = years_per_person[years_per_person["n_years"] >= 2]
print(f"patients in ≥2 years: {len(multi):,} / {years_per_person.shape[0]:,}")
print(years_per_person["n_years"].value_counts().sort_index())

# Example: drug starts mid-year Y, present again in Y+1
start_mid = pair_all[
    (pair_all["RXBEGYRX"] == pair_all["YEAR"])
    & (pair_all["drug_start_days"] < 365)
    & (pair_all["drug_start_days"] > 0)
].copy()

cont_rows = []
for r in start_mid.itertuples():
    nxt = pair_all[
        (pair_all["DUPERSID"] == r.DUPERSID)
        & (pair_all["RXNAME"] == r.RXNAME)
        & (pair_all["YEAR"] == r.YEAR + 1)
    ]
    if len(nxt):
        cont_rows.append({
            "DUPERSID": r.DUPERSID,
            "RXNAME": r.RXNAME,
            "start_year": r.YEAR,
            "start_month": r.first_month,
            "denom_start_year": r.total_days_supply,
            "drug_start_days_Y": r.drug_start_days,
            "ratio_Y": r.meps_adherence_ratio,
            "denom_next_year": nxt["total_days_supply"].iloc[0],
            "drug_start_days_Y1": nxt["drug_start_days"].iloc[0],
            "ratio_Y1": nxt["meps_adherence_ratio"].iloc[0],
        })
        if len(cont_rows) >= 8:
            break

cont_ex = pd.DataFrame(cont_rows)
print("\nMid-year start → next-year continuation examples:")
display(cont_ex)

if len(cont_ex):
    assert (cont_ex["drug_start_days_Y"] < 365).all()
    assert (cont_ex["drug_start_days_Y1"] == 365).all(), (
        "Expected next year to treat prior-year starts as full-year drug eligibility"
    )
    print("✓ next-year drug_start_days reset to 365 for continuing drugs")

# Multi-year patient panel slice
if len(multi):
    pid = multi.sort_values("n_years", ascending=False)["DUPERSID"].iloc[0]
    panel = (
        pair_all[pair_all["DUPERSID"] == pid]
        [["DUPERSID", "YEAR", "RXNAME", "total_days_supply", "drug_start_days",
          "RXBEGYRX", "meps_adherence_ratio", "is_adherent", "AGE"]]
        .sort_values(["YEAR", "RXNAME"])
    )
    print(f"\nPanel slice for DUPERSID={pid} ({panel['YEAR'].nunique()} years):")
    display(panel.head(20))
    assert panel.groupby("YEAR").ngroups >= 2
    print("✓ same DUPERSID retains separate YEAR rows with year-specific denoms")

patients in ≥2 years: 8,847 / 18,673
n_years
1    9826
2    7676
3    1171
Name: count, dtype: int64

Mid-year start → next-year continuation examples:


,DUPERSID,RXNAME,start_year,start_month,denom_start_year,drug_start_days_Y,ratio_Y,denom_next_year,drug_start_days_Y1,ratio_Y1
0,2320018101,AMLODIPINE,2020,11.0,61,61,100.000000,365,365,49.315068
1,2320018102,LISINOPRIL,2020,4.0,275,275,32.727273,365,365,98.630137
2,2320050101,LISINOPRIL,2020,7.0,184,184,97.826087,365,365,73.972603
3,2320135101,ELIQUIS,2020,3.0,306,306,88.235294,365,365,24.657534
4,2320194101,SERTRALINE,2020,12.0,31,31,100.000000,365,365,73.972603
5,2320194101,TRAZODONE,2020,12.0,31,31,96.774194,365,365,49.315068
6,2320280103,SERTRALINE,2020,3.0,306,306,78.431373,365,365,8.219178
7,2320280103,ATOMOXETINE,2020,11.0,61,61,49.180328,365,365,16.438356


✓ next-year drug_start_days reset to 365 for continuing drugs

Panel slice for DUPERSID=2463991102 (3 years):


,DUPERSID,YEAR,RXNAME,total_days_supply,drug_start_days,RXBEGYRX,meps_adherence_ratio,is_adherent,AGE
9877,2463991102,2020,LISINOPRIL,365,365,2018,100.000000,1,66
28947,2463991102,2021,LISINOPRIL,365,365,2018,100.000000,1,67
46172,2463991102,2022,LISINOPRIL,365,365,2018,52.054795,0,68


✓ same DUPERSID retains separate YEAR rows with year-specific denoms


## Step E — Merge all years (column-align one-hots)

Patient–year frames can have different `RX_*` / `ICD_*` columns. We outer-align and fill missing indicators with **0** (drug/condition not observed that year).

In [15]:
frames = []
for y in YEARS:
    df = model_by_year[y].copy()
    assert "YEAR" in df.columns
    frames.append(df)

model_df_all = pd.concat(frames, ignore_index=True, sort=False)

# One-hot columns that appear in only some years → 0
oh_cols = [c for c in model_df_all.columns if c.startswith(("RX_", "ICD_"))]
model_df_all[oh_cols] = model_df_all[oh_cols].fillna(0).astype(int)

# Recompute counts after alignment (safe if already present)
rx_cols = [c for c in model_df_all.columns if c.startswith("RX_")]
icd_cols = [c for c in model_df_all.columns if c.startswith("ICD_")]
model_df_all["n_drugs"] = model_df_all[rx_cols].sum(axis=1)
model_df_all["n_conditions"] = model_df_all[icd_cols].sum(axis=1)

print(f"merged patient-year rows: {len(model_df_all):,}")
print(f"unique DUPERSID: {model_df_all['DUPERSID'].nunique():,}")
print(f"unique (DUPERSID, YEAR): {model_df_all.groupby(['DUPERSID','YEAR']).ngroups:,}")
print("rows by YEAR:")
print(model_df_all["YEAR"].value_counts().sort_index())
print(f"RX dummy cols: {len(rx_cols)} | ICD dummy cols: {len(icd_cols)}")
print("is_adherent by YEAR:")
display(model_df_all.groupby("YEAR")["is_adherent"].mean().rename("adherent_rate"))

# Keys must be unique at patient-year grain
dup = model_df_all.duplicated(["DUPERSID", "YEAR"]).sum()
assert dup == 0, f"duplicate (DUPERSID, YEAR) keys: {dup}"
print("✓ (DUPERSID, YEAR) is unique")

model_df_all.head()

merged patient-year rows: 28,691
unique DUPERSID: 18,673
unique (DUPERSID, YEAR): 28,691
rows by YEAR:
YEAR
2020    7718
2021    8377
2022    6770
2023    5826
Name: count, dtype: int64
RX dummy cols: 427 | ICD dummy cols: 74
is_adherent by YEAR:


YEAR
2020    0.536279
2021    0.513668
2022    0.505465
2023    0.494851
Name: adherent_rate, dtype: float64

✓ (DUPERSID, YEAR) is unique


,DUPERSID,YEAR,AGE,FAMINC,is_adherent,INSCOV_PRIVATE,INSCOV_PUBLIC,INSCOV_UNINSURED,MALE,FEMALE,WHITE,BLACK,AMER_INDIAN,ASIAN_INDIAN,CHINESE,FILIPINO,PMED_DELAY_COST,NO_PMED_DELAY_COST,CARE_DELAY_COST,NO_CARE_DELAY_COST,...,RX_PROGESTERONE 225,RX_REMERON,RX_REPATHA,RX_REPATHA INJ,RX_SEMAGLUTIDE,RX_STIMULANT LX,RX_TAMSULOSIN HCL,RX_TESTOSTERONE (YAM) 25MG,RX_TESTOSTERONE 0.5MG/1ML,RX_TESTOSTERONE 160MG TROCHE,RX_TESTOSTERONE 50MG MINI,RX_TESTOSTERONE CREAM 1MG/ML,RX_TESTOSTERONE CYPIONATE IM,RX_THALITONE,RX_TIMOPTIC,RX_TRESIBA FLEX INJ,RX_VENLAFAXINE HCL CAP ER,RX_ZEPBOUND,RX_ZETIA,ICD_U09
0,2320005102,2020,84,17000,1,0,1,0,1,0,1,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2320013101,2020,85,18960,0,0,1,0,0,1,1,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2320018101,2020,58,28920,1,0,1,0,0,1,1,0,0,0,0,0,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,2320018102,2020,72,28920,1,0,1,0,1,0,1,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,2320019102,2020,47,98788,1,1,0,0,0,1,1,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## Step F — Write merged panel

In [16]:
tables_all, _ = all_years_output_dirs()
tables_all.mkdir(parents=True, exist_ok=True)

out_parquet = tables_all / "model_df_all_years.parquet"
out_pair = tables_all / "model_df_pair_all_years.parquet"

if "AGE" in model_df_all.columns:
    model_df_all["AGE"] = model_df_all["AGE"].astype("string")
if "AGE" in pair_all.columns:
    pair_all["AGE"] = pair_all["AGE"].astype("string")

model_df_all.to_parquet(out_parquet, index=False)
pair_all.to_parquet(out_pair, index=False)

print(f"wrote {out_parquet} ({len(model_df_all):,} rows, {model_df_all.shape[1]} cols)")
print(f"wrote {out_pair} ({len(pair_all):,} rows, {pair_all.shape[1]} cols)")

# Also keep per-year paths visible
for y in YEARS:
    t, _ = output_dirs(y)
    print(f"  {y}: {t / 'model_df.parquet'}")

wrote /Users/friana/Medical Adherence/Notebooks/MEPS/output/all_years/tables/model_df_all_years.parquet (28,691 rows, 530 cols)
wrote /Users/friana/Medical Adherence/Notebooks/MEPS/output/all_years/tables/model_df_pair_all_years.parquet (79,029 rows, 39 cols)
  2020: /Users/friana/Medical Adherence/Notebooks/MEPS/output/2020/tables/model_df.parquet
  2021: /Users/friana/Medical Adherence/Notebooks/MEPS/output/2021/tables/model_df.parquet
  2022: /Users/friana/Medical Adherence/Notebooks/MEPS/output/2022/tables/model_df.parquet
  2023: /Users/friana/Medical Adherence/Notebooks/MEPS/output/2023/tables/model_df.parquet


## Checklist (what this notebook guarantees)

1. Each year’s adherence uses **that year’s** fills and PSTATS window only.
2. Multi-year patients appear as **multiple rows** keyed by `(DUPERSID, YEAR)`.
3. Mid-year drug starts shorten **only the start year**; the next year resets `drug_start_days` to 365.
4. No cross-year days-supply carryover.
5. Merged one-hots are column-aligned with missing year–drug/condition cells = 0.
6. Outputs written under `Notebooks/MEPS/output/`.

## Step G — Same-drug continuity for patients in >2 years

Among patients present in **more than 2 years** (`n_years > 2`), how many keep the **same `RXNAME` in every year** they appear in the panel?

- **≥1 continuous drug:** at least one drug shows up in *all* of that patient’s survey years.
- **Identical regimen:** the full set of `RXNAME`s is the same in every year (stricter).

In [17]:
# Person–year–drug grain (pair rows can repeat ICD; drop that for continuity)
pyd = pair_all[["DUPERSID", "YEAR", "RXNAME"]].drop_duplicates()

years_per_person = (
    pyd.groupby("DUPERSID")["YEAR"].nunique().rename("n_years")
)
multi_gt2 = years_per_person[years_per_person > 2]
print(f"patients in >2 years: {len(multi_gt2):,} / {years_per_person.shape[0]:,}")
print(years_per_person.value_counts().sort_index().to_string())

sub = pyd[pyd["DUPERSID"].isin(multi_gt2.index)].copy()

# Years each (person, drug) appears vs years the person is in the survey
drug_years = (
    sub.groupby(["DUPERSID", "RXNAME"])["YEAR"]
    .nunique()
    .rename("n_drug_years")
    .reset_index()
)
drug_years = drug_years.merge(
    multi_gt2.rename("n_person_years"),
    left_on="DUPERSID",
    right_index=True,
)
drug_years["continuous"] = (
    drug_years["n_drug_years"] == drug_years["n_person_years"]
)

has_any_continuous = drug_years.groupby("DUPERSID")["continuous"].any()
n_any = int(has_any_continuous.sum())
n_none = int((~has_any_continuous).sum())

# Stricter: identical RXNAME set every year
identical_flags = []
for pid, g in sub.groupby("DUPERSID"):
    year_sets = {frozenset(names) for _, names in g.groupby("YEAR")["RXNAME"]}
    identical_flags.append(len(year_sets) == 1)
n_identical = int(sum(identical_flags))

print(
    f"\n≥1 drug present in ALL survey years: "
    f"{n_any:,} / {len(multi_gt2):,} ({100 * n_any / len(multi_gt2):.1f}%)"
)
print(
    f"no drug continuous across all years: "
    f"{n_none:,} ({100 * n_none / len(multi_gt2):.1f}%)"
)
print(
    f"identical RXNAME set every year: "
    f"{n_identical:,} / {len(multi_gt2):,} ({100 * n_identical / len(multi_gt2):.1f}%)"
)

n_cont_drugs = int(drug_years["continuous"].sum())
print(
    f"\ncontinuous person–drug pairs: "
    f"{n_cont_drugs:,} / {len(drug_years):,}"
)
print("\n# continuous drugs per patient (among >2-year patients):")
print(
    drug_years.groupby("DUPERSID")["continuous"]
    .sum()
    .astype(int)
    .value_counts()
    .sort_index()
    .to_string()
)

# Example continuous drugs
cont_examples = (
    drug_years.loc[drug_years["continuous"]]
    .merge(
        sub.groupby("DUPERSID")["YEAR"]
        .agg(years=lambda s: sorted(s.unique()))
        .reset_index(),
        on="DUPERSID",
    )
    .sort_values(["DUPERSID", "RXNAME"])
    .head(12)
)
print("\nExample continuous (DUPERSID, RXNAME) pairs:")
display(cont_examples[["DUPERSID", "RXNAME", "n_drug_years", "n_person_years", "years"]])

patients in >2 years: 1,171 / 18,673
n_years
1    9826
2    7676
3    1171

≥1 drug present in ALL survey years: 996 / 1,171 (85.1%)
no drug continuous across all years: 175 (14.9%)
identical RXNAME set every year: 242 / 1,171 (20.7%)

continuous person–drug pairs: 2,072 / 5,111

# continuous drugs per patient (among >2-year patients):
continuous
0    175
1    454
2    260
3    141
4     73
5     40
6     18
7      7
8      1
9      2

Example continuous (DUPERSID, RXNAME) pairs:


,DUPERSID,RXNAME,n_drug_years,n_person_years,years
0,2460002101,JANUVIA,3,3,"[2020, 2021, 2022]"
1,2460002101,ROSUVASTATIN,3,3,"[2020, 2021, 2022]"
2,2460006101,LOSARTAN POT,3,3,"[2020, 2021, 2022]"
3,2460010101,LAMOTRIGINE,3,3,"[2020, 2021, 2022]"
4,2460040101,TRIAMT/HCTZ,3,3,"[2020, 2021, 2022]"
5,2460050101,CARVEDILOL,3,3,"[2020, 2021, 2022]"
6,2460050101,ISOSORB MONO,3,3,"[2020, 2021, 2022]"
7,2460053101,LISINOP/HCTZ,3,3,"[2020, 2021, 2022]"
8,2460085101,ELIQUIS,3,3,"[2020, 2021, 2022]"
9,2460085101,LISINOP/HCTZ,3,3,"[2020, 2021, 2022]"


## Step H — Same-drug continuity for patients in >1 year (+ denom / gap note)

**Continuity question:** among patients in **more than 1 year** (`n_years > 1`), how many keep the **same `RXNAME` in every year they appear** in this chronic-drug panel?

**How the adherence denominator works (leave / return):**

| Scope | What we do |
|---|---|
| **Within a year** | Yes — PSTATS + BEGRF/ENDRF set eligible days. Mid-year leavers, joiners, death, and round nonresponse shrink `total_days_supply`. Full-year (`PSTATS` all 11) → 365. |
| **Between years** | Years are **independent**. A 2020 row only uses 2020 PSTATS; a 2022 row only uses 2022 PSTATS. We do **not** stitch a multi-year window or punish Year N+1 for absence in Year N. |
| **Leave then return** | Gap years simply have **no person–year row** in the panel (for this build: no chronic Rx pair that year). Continuity below only requires the drug in the years the person *does* appear — not in the gap. |

So: we **do** check in-survey / eligible days **within each year**; we do **not** require continuous survey membership across calendar years for either adherence or the continuity counts below.

**Caveat:** “Present in year Y” here means present in `pair_all` (chronic drug–condition pairs), not raw MEPS person-file membership with zero chronic fills.

In [18]:
# --- Continuity among patients in >1 year (panel years, not calendar span) ---
pyd_h = pair_all[["DUPERSID", "YEAR", "RXNAME"]].drop_duplicates()

years_per_person_h = (
    pyd_h.groupby("DUPERSID")["YEAR"].nunique().rename("n_years")
)
multi_gt1 = years_per_person_h[years_per_person_h > 1]
print(f"patients in >1 year: {len(multi_gt1):,} / {years_per_person_h.shape[0]:,}")
print(years_per_person_h.value_counts().sort_index().to_string())

sub_h = pyd_h[pyd_h["DUPERSID"].isin(multi_gt1.index)].copy()

person_year_lists = (
    sub_h.groupby("DUPERSID")["YEAR"]
    .agg(lambda s: sorted(s.unique()))
    .rename("years")
)


def _has_calendar_gap(years) -> bool:
    """True if person skips a calendar year between first and last panel year."""
    ys = list(years)
    return any(ys[i + 1] - ys[i] > 1 for i in range(len(ys) - 1))


gap_flag = person_year_lists.map(_has_calendar_gap).rename("has_gap")
n_gap = int(gap_flag.sum())
n_consec = int((~gap_flag).sum())
print(
    f"\ncalendar gap in panel years (e.g. 2020+2022, no 2021): "
    f"{n_gap:,} / {len(multi_gt1):,} ({100 * n_gap / len(multi_gt1):.1f}%)"
)
print(f"consecutive panel years only: {n_consec:,}")

drug_years_h = (
    sub_h.groupby(["DUPERSID", "RXNAME"])["YEAR"]
    .nunique()
    .rename("n_drug_years")
    .reset_index()
)
drug_years_h = drug_years_h.merge(
    multi_gt1.rename("n_person_years"),
    left_on="DUPERSID",
    right_index=True,
)
# Continuous = drug appears in EVERY year this person is in the panel
# (gap years are ignored — person was not in panel that year)
drug_years_h["continuous"] = (
    drug_years_h["n_drug_years"] == drug_years_h["n_person_years"]
)

has_any_h = drug_years_h.groupby("DUPERSID")["continuous"].any()
n_any_h = int(has_any_h.sum())
n_none_h = int((~has_any_h).sum())

identical_flags_h = []
for _, g in sub_h.groupby("DUPERSID"):
    year_sets = {frozenset(names) for _, names in g.groupby("YEAR")["RXNAME"]}
    identical_flags_h.append(len(year_sets) == 1)
n_identical_h = int(sum(identical_flags_h))

print(
    f"\n≥1 drug present in ALL panel years: "
    f"{n_any_h:,} / {len(multi_gt1):,} ({100 * n_any_h / len(multi_gt1):.1f}%)"
)
print(
    f"no drug continuous across all panel years: "
    f"{n_none_h:,} ({100 * n_none_h / len(multi_gt1):.1f}%)"
)
print(
    f"identical RXNAME set every panel year: "
    f"{n_identical_h:,} / {len(multi_gt1):,} ({100 * n_identical_h / len(multi_gt1):.1f}%)"
)

# Split by gap vs consecutive
cont_by_gap = (
    has_any_h.to_frame("has_continuous_drug")
    .join(gap_flag)
    .groupby("has_gap")["has_continuous_drug"]
    .agg(n="sum", total="count")
)
cont_by_gap.index = cont_by_gap.index.map({False: "consecutive years", True: "gapped years"})
print("\n≥1 continuous drug by gap status:")
for label, row in cont_by_gap.iterrows():
    print(
        f"  {label}: {int(row['n']):,} / {int(row['total']):,} "
        f"({100 * row['n'] / row['total']:.1f}%)"
    )

n_cont_drugs_h = int(drug_years_h["continuous"].sum())
print(
    f"\ncontinuous person–drug pairs: "
    f"{n_cont_drugs_h:,} / {len(drug_years_h):,}"
)

# --- Denominator within each panel year (PSTATS window, not cross-year) ---
py_denom = (
    pair_all[pair_all["DUPERSID"].isin(multi_gt1.index)][
        ["DUPERSID", "YEAR", "total_days_supply"]
    ]
    .drop_duplicates()
)
print("\n--- Adherence denom among >1-year person–years ---")
print(
    f"person–years: {len(py_denom):,}  |  "
    f"denom==365: {(py_denom['total_days_supply'] == 365).sum():,}  |  "
    f"denom<365 (partial / drug-start clamp): "
    f"{(py_denom['total_days_supply'] < 365).sum():,}"
)
print(py_denom["total_days_supply"].describe().round(1).to_string())

# Gap examples + continuous examples
if n_gap:
    gap_ex = person_year_lists[gap_flag].head(8).reset_index()
    print("\nExample gapped panel-year sequences (leave/return or no chronic Rx that year):")
    display(gap_ex)

cont_ex_h = (
    drug_years_h.loc[drug_years_h["continuous"]]
    .merge(person_year_lists.reset_index(), on="DUPERSID")
    .merge(gap_flag.reset_index(), on="DUPERSID")
    .sort_values(["has_gap", "DUPERSID", "RXNAME"], ascending=[False, True, True])
    .head(12)
)
print("\nExample continuous (DUPERSID, RXNAME) pairs (gapped first if any):")
display(
    cont_ex_h[
        ["DUPERSID", "RXNAME", "n_drug_years", "n_person_years", "years", "has_gap"]
    ]
)

patients in >1 year: 8,847 / 18,673
n_years
1    9826
2    7676
3    1171

calendar gap in panel years (e.g. 2020+2022, no 2021): 99 / 8,847 (1.1%)
consecutive panel years only: 8,748

≥1 drug present in ALL panel years: 7,999 / 8,847 (90.4%)
no drug continuous across all panel years: 848 (9.6%)
identical RXNAME set every panel year: 3,286 / 8,847 (37.1%)

≥1 continuous drug by gap status:
  consecutive years: 7,940 / 8,748 (90.8%)
  gapped years: 59 / 99 (59.6%)

continuous person–drug pairs: 17,878 / 31,784

--- Adherence denom among >1-year person–years ---
person–years: 24,366  |  denom==365: 17,889  |  denom<365 (partial / drug-start clamp): 6,473
count    24366.0
mean       317.7
std         92.8
min         31.0
25%        334.0
50%        365.0
75%        365.0
max        366.0

Example gapped panel-year sequences (leave/return or no chronic Rx that year):


,DUPERSID,years
0,2460128101,"[2020, 2022]"
1,2460183101,"[2020, 2022]"
2,2460247102,"[2020, 2022]"
3,2460398101,"[2020, 2022]"
4,2460466101,"[2020, 2022]"
5,2460661101,"[2020, 2022]"
6,2460720102,"[2020, 2022]"
7,2460880102,"[2020, 2022]"



Example continuous (DUPERSID, RXNAME) pairs (gapped first if any):


,DUPERSID,RXNAME,n_drug_years,n_person_years,years,has_gap
3728,2460183101,ANASTROZOLE,2,2,"[2020, 2022]",True
3729,2460183101,LISINOPRIL,2,2,"[2020, 2022]",True
3742,2460247102,BRIMONIDINE,2,2,"[2020, 2022]",True
3796,2460466101,LOSARTAN POT,2,2,"[2020, 2022]",True
3885,2460720102,BUPROPION,2,2,"[2020, 2022]",True
3932,2460905102,EZETIMIBE,2,2,"[2020, 2022]",True
3961,2461001101,LEVOTHYROXIN,2,2,"[2020, 2022]",True
3972,2461069101,ATORVASTATIN,2,2,"[2020, 2022]",True
4001,2461178101,HYDRALAZINE,2,2,"[2020, 2022]",True
4083,2461412101,LOSARTAN POT,2,2,"[2020, 2022]",True


## Step I — Denom=0 bug fix (PSTATS=12)

**Root cause:** `2791970102` (and every other former denom=0 case) had `PSTATS = 12/12/12` with full `BEGRF`/`ENDRF` windows and Rx fills (`RXBEGYRX=2018` for LOSARTAN POT). Code `12` was incorrectly listed in `NO_COVERAGE_STATUSES`, so the pipeline zeroed eligible days → `total_valid_days=0` and `meps_adherence_ratio=NaN`.

**MEPS Table 8 (h251doc):** code 12 = FT military in the RU, out-of-scope for *national civilian estimates*, but it still has applicable reference dates (same rule as code 11) and skips **no** instrument sections. True no-coverage codes (0, 21, 24, 43, 62–64, 72–74, 81) have “Inapplicable” reference dates.

**Fix in `app/clean_meps.py`:** move `12` into `FULL_ROUND_STATUSES`; recompute denoms/numerators on saved year tables. After fix: **0** person–drug rows with denom 0.

In [19]:
# Verify PSTATS=12 patient after denom fix (reload from repaired parquets)
PID = 2791970102
check = (
    pd.concat(
        [
            pd.read_parquet(output_dirs(y)[0] / "new_grouped_merge_df_chronic_drugs.parquet").assign(YEAR=y)
            for y in YEARS
        ],
        ignore_index=True,
    )
)
row = check[check["DUPERSID"] == PID][
    [
        "DUPERSID", "YEAR", "RXNAME", "RXDAYSUP", "drug_start_days",
        "total_days_supply", "total_valid_days", "meps_adherence_ratio",
        "participation_type", "coverage_notes",
    ]
].sort_values("YEAR")
print(f"denom==0 rows across years: {(check['total_days_supply'] == 0).sum():,}")
display(row)
assert (row["total_days_supply"] > 0).all()
assert (row["coverage_notes"] == "full_year_all_rounds_12").all()
print("✓ 2791970102 now has full-year denom; numerator = capped RXDAYSUP")

denom==0 rows across years: 0


,DUPERSID,YEAR,RXNAME,RXDAYSUP,drug_start_days,total_days_supply,total_valid_days,meps_adherence_ratio,participation_type,coverage_notes


✓ 2791970102 now has full-year denom; numerator = capped RXDAYSUP


## Step J — Drop excluded PSTATS + denom from round BEGRF/ENDRF

**Dropped** if *any* of `PSTATS31/42/53` ∈  
`{12,13,23,24,31,32,33,35,36,43,61,62,63,64,72,73,74,81}`:

| Year | Persons dropped | Person–drug rows | Pair rows | Model rows |
|------|----------------:|-----------------:|----------:|-----------:|
| 2020 | 159 / 7,954 | 413 | 260 | 99 |
| 2021 | 164 / 8,631 | 471 | 277 | 94 |
| 2022 | 105 / 6,932 | 243 | 125 | 56 |
| 2023 | 84 / 5,968 | 193 | 94 | 37 |
| **Total** | **512** | **1,320** | **756** | **286** |

(Mostly code **31** death; also 12/13/32.) The prior ~18 denom=0 cases were PSTATS **12** and are in this drop.

**Denominator (kept persons):** union of each in-scope round’s `BEGRF`→`ENDRF` month-year window, clamped to the calendar year; then `min(that, drug_start_days)`. Skipped rounds (`PSTATS=-1`) add no days. After drop + recompute: **0** rows with denom 0; min denom among kept ≈ 31 days.